# FASE 3: ENTRENAMIENTO Y EVALUACIÓN DE MODELOS DE MACHINE LEARNING (LOCAL)

## 📌 Contexto de Negocio
Para Telco, retener a un cliente existente es 5 veces más económico que adquirir uno nuevo a través de campañas de marketing. En las fases previas (EDA y Pipeline), identificamos los factores clave que impulsan la fuga de clientes (*Churn*) y transformamos esos datos crudos en un vector de características optimizado. En esta fase, utilizaremos esos datos para entrenar algoritmos predictivos que funcionen como un "radar de riesgo", permitiendo al equipo de Marketing actuar de forma proactiva y dirigida hacia los clientes con alta probabilidad de abandono.

## 🎯 Objetivo de este Notebook
Entrenar y comparar **dos modelos de clasificación binaria** (Regresión Logística y Random Forest) usando `pyspark.ml`, y evaluarlos no solo con ROC AUC sino con las métricas que realmente importan para el negocio: **Recall, Precision y F1-Score sobre la clase Churn=1**, ya que la meta del proyecto es lograr **Recall > 80%** en la detección de clientes en riesgo.

## ⚠️ Cambio respecto a la versión anterior
En la Fase 1 (EDA) se decidió imputar los valores nulos de `TotalCharges` con **0.0**, justificado en que corresponden a clientes nuevos con `tenure = 0` que aún no han generado facturación. Esta versión del notebook **alinea la limpieza de datos con esa decisión del EDA** (antes se usaba la mediana, lo cual no tenía sentido de negocio para clientes recién ingresados).

## 📊 Estructura del Experimento
1. **Inicialización del Motor:** Sesión local de PySpark con 4GB de RAM.
2. **Pipeline Integrado:** Ingesta, limpieza (alineada con EDA), indexación categórica y vectorización.
3. **División Estratégica (Train/Test Split):** 70% entrenamiento / 30% prueba.
4. **Entrenamiento de Modelos:** Regresión Logística (baseline) y Random Forest (modelo de comparación, tal como se definió en el stack tecnológico del README).
5. **Evaluación Completa:** ROC AUC, Accuracy, Precision, Recall y F1-Score sobre la clase positiva (Churn), además de matriz de confusión.
6. **Ajuste de Umbral (Threshold Tuning):** Dado que el dataset está desbalanceado (~27% Churn), se prueba bajar el umbral de decisión para intentar alcanzar el Recall objetivo (>80%) y se analiza el trade-off con Precision.
7. **Conclusión Comparativa:** Tabla final indicando qué modelo/configuración cumple la meta del proyecto.
8. **Importancia de Variables:** Extracción del ranking de factores que más influyen en el Churn según Random Forest, para que el dashboard explique *por qué* se van los clientes, no solo *cuántos*.
9. **Exportación para el Dashboard:** Se selecciona automáticamente el modelo con mejor Recall y se exportan sus predicciones junto al `customerID` (antes se perdía en el ensamblado de features), más el ranking de variables, en archivos listos para que la Fase 4 (Dashboard) los consuma.

## 1. Inicialización y configuración del entorno de ML

In [2]:
# ==============================================================================
# 1. INICIALIZACIÓN Y CONFIGURACIÓN DEL ENTORNO DE ML
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Iniciar o recuperar la sesión local de Spark
spark = SparkSession.builder \
    .appName("TelcoChurn_MachineLearning") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Sesión de Spark lista para Machine Learning. Versión: {spark.version}")

Sesión de Spark lista para Machine Learning. Versión: 3.5.0


## 2. Pipeline integrado y división de datos (Train/Test Split)

In [3]:
# ==============================================================================
# 2. PIPELINE INTEGRADO Y DIVISIÓN DE DATOS (TRAIN/TEST SPLIT)
# ==============================================================================
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler

# --- PASO 1: Ingesta Local ---
csv_path = "../data/sample/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(csv_path)

# --- PASO 2: Limpieza de TotalCharges (alineada con la conclusión del EDA) ---
# Los espacios en blanco corresponden a clientes con tenure = 0 (recién ingresados,
# todavía no facturados). El valor de negocio correcto para ellos es 0.0, no la mediana.
df_cleaned = df.withColumn("TotalCharges", col("TotalCharges").cast("double"))
df_cleaned = df_cleaned.na.fill({"TotalCharges": 0.0})

# --- PASO 3: Indexación de Textos ---
columnas_categoricas = [item[0] for item in df_cleaned.dtypes if item[1] == 'string' and item[0] != 'customerID']
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in columnas_categoricas]
df_indexed = df_cleaned
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

# --- PASO 4: Ensamblado Vectorial ---
columnas_features = [
    "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
    "gender_index", "Partner_index", "Dependents_index", "PhoneService_index",
    "MultipleLines_index", "InternetService_index", "OnlineSecurity_index",
    "OnlineBackup_index", "DeviceProtection_index", "TechSupport_index",
    "StreamingTV_index", "StreamingMovies_index", "Contract_index",
    "PaperlessBilling_index", "PaymentMethod_index"
]
assembler = VectorAssembler(inputCols=columnas_features, outputCol="features", handleInvalid="skip")
df_final = assembler.transform(df_indexed)

# --- PASO 5: Selección final con label binario (int), conservando customerID ---
# OJO: antes solo se seleccionaba "features" y "label", lo cual perdía el customerID.
# Lo conservamos para poder identificar después a QUÉ clientes corresponde cada predicción
# (imprescindible para el dashboard: no sirve un dashboard que diga "27% en riesgo" sin decir quiénes).
df_vectorizado = df_final.select("customerID", "features", col("Churn_index").cast("int").alias("label"))

# --- PASO 6: División de Datos (70% Train, 30% Test) ---
train_data, test_data = df_vectorizado.randomSplit([0.7, 0.3], seed=42)

print(f"Filas en entrenamiento: {train_data.count()}")
print(f"Filas en prueba: {test_data.count()}")

# Verificar el balance de clases (importante para interpretar Recall más adelante)
print("\nDistribución de clases en el set completo:")
df_vectorizado.groupBy("label").count().show()

Filas en entrenamiento: 5036
Filas en prueba: 2007

Distribución de clases en el set completo:
+-----+-----+
|label|count|
+-----+-----+
|    1| 1869|
|    0| 5174|
+-----+-----+



## 3. Modelo 1: Regresión Logística (baseline)

In [4]:
# ==============================================================================
# 3. ENTRENAMIENTO: REGRESIÓN LOGÍSTICA BINARIA
# ==============================================================================
from pyspark.ml.classification import LogisticRegression

print("Entrenando Regresión Logística...")

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=10,
    family="binomial"
)

lr_model = lr.fit(train_data)
predictions_lr = lr_model.transform(test_data)

print("¡Regresión Logística entrenada con éxito!")
predictions_lr.select("label", "rawPrediction", "probability", "prediction").show(5, truncate=False)

Entrenando Regresión Logística...
¡Regresión Logística entrenada con éxito!
+-----+------------------------------------------+----------------------------------------+----------+
|label|rawPrediction                             |probability                             |prediction|
+-----+------------------------------------------+----------------------------------------+----------+
|1    |[-0.7014432705811211,0.7014432705811211]  |[0.33149231387594863,0.6685076861240513]|1.0       |
|0    |[2.576445478707396,-2.576445478707396]    |[0.9293301797546387,0.07066982024536128]|0.0       |
|0    |[0.41697210084704284,-0.41697210084704284]|[0.6027584736461157,0.3972415263538843] |0.0       |
|0    |[3.1953366346137813,-3.1953366346137813]  |[0.960658408885381,0.03934159111461899] |0.0       |
|0    |[2.5261981760841765,-2.5261981760841765]  |[0.9259581233118249,0.07404187668817508]|0.0       |
+-----+------------------------------------------+----------------------------------------+---------

## 4. Modelo 2: Random Forest (comparación)
El README del proyecto define Random Forest o GBT Classifier como el algoritmo objetivo en el stack tecnológico. Se entrena aquí para comparar contra la Regresión Logística antes de decidir cuál pasa al dashboard.

In [5]:
# ==============================================================================
# 4. ENTRENAMIENTO: RANDOM FOREST CLASSIFIER
# ==============================================================================
from pyspark.ml.classification import RandomForestClassifier

print("Entrenando Random Forest...")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_model = rf.fit(train_data)
predictions_rf = rf_model.transform(test_data)

print("¡Random Forest entrenado con éxito!")
predictions_rf.select("label", "rawPrediction", "probability", "prediction").show(5, truncate=False)

Entrenando Random Forest...
¡Random Forest entrenado con éxito!
+-----+--------------------------------------+----------------------------------------+----------+
|label|rawPrediction                         |probability                             |prediction|
+-----+--------------------------------------+----------------------------------------+----------+
|1    |[38.15895523254414,61.841044767455834]|[0.3815895523254415,0.6184104476745585] |1.0       |
|0    |[95.33346585596277,4.666534144037203] |[0.953334658559628,0.04666534144037204] |0.0       |
|0    |[62.97134369384262,37.028656306157366]|[0.6297134369384263,0.3702865630615737] |0.0       |
|0    |[96.4092196170782,3.5907803829217673] |[0.9640921961707823,0.03590780382921769]|0.0       |
|0    |[94.27339312779726,5.726606872202759] |[0.9427339312779723,0.05726606872202757]|0.0       |
+-----+--------------------------------------+----------------------------------------+----------+
only showing top 5 rows



## 5. Evaluación completa: ROC AUC, Precision, Recall, F1 y Matriz de Confusión
Se evalúa la clase positiva (`label = 1`, Churn) porque es la que el negocio necesita detectar. ROC AUC sola no es suficiente: con un dataset desbalanceado (~27% Churn) un modelo puede tener AUC alto y aun así fallar en Recall.

In [6]:
# ==============================================================================
# 5. FUNCIÓN DE EVALUACIÓN COMPLETA
# ==============================================================================
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def evaluar_modelo(predictions, nombre_modelo):
    print(f"\n{'='*55}")
    print(f"EVALUACIÓN: {nombre_modelo}")
    print(f"{'='*55}")

    # --- ROC AUC ---
    evaluator_auc = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction", labelCol="label", metricName="areaUnderROC"
    )
    roc_auc = evaluator_auc.evaluate(predictions)

    # --- Precision, Recall y F1 sobre la clase Churn=1 ---
    evaluator_mc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

    recall_churn = evaluator_mc.evaluate(predictions, {evaluator_mc.metricName: "recallByLabel", evaluator_mc.metricLabel: 1.0})
    precision_churn = evaluator_mc.evaluate(predictions, {evaluator_mc.metricName: "precisionByLabel", evaluator_mc.metricLabel: 1.0})
    f1_churn = evaluator_mc.evaluate(predictions, {evaluator_mc.metricName: "fMeasureByLabel", evaluator_mc.metricLabel: 1.0})
    accuracy = evaluator_mc.evaluate(predictions, {evaluator_mc.metricName: "accuracy"})

    print(f"ROC AUC:               {roc_auc:.4f}")
    print(f"Accuracy:              {accuracy:.4f}")
    print(f"Precision (Churn=1):   {precision_churn:.4f}")
    print(f"Recall (Churn=1):      {recall_churn:.4f}   <-- meta del proyecto: > 0.80")
    print(f"F1-Score (Churn=1):    {f1_churn:.4f}")

    print("\nMatriz de Confusión:")
    predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

    cumple_meta = "✅ SÍ" if recall_churn > 0.80 else "❌ NO"
    print(f"¿Cumple Recall > 80%?  {cumple_meta}")

    return {
        "modelo": nombre_modelo,
        "roc_auc": roc_auc,
        "accuracy": accuracy,
        "precision": precision_churn,
        "recall": recall_churn,
        "f1": f1_churn
    }

resultado_lr = evaluar_modelo(predictions_lr, "Regresión Logística (umbral 0.5)")
resultado_rf = evaluar_modelo(predictions_rf, "Random Forest (umbral 0.5)")


EVALUACIÓN: Regresión Logística (umbral 0.5)
ROC AUC:               0.8486
Accuracy:              0.8112
Precision (Churn=1):   0.6837
Recall (Churn=1):      0.5302   <-- meta del proyecto: > 0.80
F1-Score (Churn=1):    0.5972

Matriz de Confusión:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1347|
|    0|       1.0|  130|
|    1|       0.0|  249|
|    1|       1.0|  281|
+-----+----------+-----+

¿Cumple Recall > 80%?  ❌ NO

EVALUACIÓN: Random Forest (umbral 0.5)
ROC AUC:               0.8553
Accuracy:              0.8087
Precision (Churn=1):   0.6941
Recall (Churn=1):      0.4925   <-- meta del proyecto: > 0.80
F1-Score (Churn=1):    0.5762

Matriz de Confusión:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1362|
|    0|       1.0|  115|
|    1|       0.0|  269|
|    1|       1.0|  261|
+-----+----------+-----+

¿Cumple Recall > 80%?  ❌ NO


## 6. Ajuste de umbral (Threshold Tuning) para subir el Recall
Por defecto, ambos modelos clasifican como Churn solo si la probabilidad supera 0.5. Como la clase Churn es minoritaria, esto suele dejar el Recall por debajo de la meta. Bajar el umbral hace que el modelo "se anime" a marcar más clientes como riesgo, subiendo el Recall a cambio de algo de Precision — un trade-off aceptable para una campaña de retención, donde es más barato contactar a un cliente de más que perder a uno que se va.

In [7]:
# ==============================================================================
# 6. RECLASIFICACIÓN CON UMBRAL MÁS BAJO (EJEMPLO: 0.35)
# ==============================================================================
# NOTA: usamos vector_to_array (función nativa de Spark) en vez de un UDF de Python.
# Un UDF normal sobre la columna "probability" (tipo Vector) puede romperse con
# "PickleException: expected zero arguments for construction of ClassDict (DenseVector)"
# en ciertas versiones de PySpark, porque ese tipo no serializa bien entre Java y Python.
# vector_to_array evita el problema porque se ejecuta dentro del motor de Spark.
from pyspark.ml.functions import vector_to_array

UMBRAL = 0.35

predictions_lr_ajustado = predictions_lr.withColumn(
    "prob_churn", vector_to_array(col("probability"))[1]
)
predictions_lr_ajustado = predictions_lr_ajustado.withColumn(
    "prediction", (col("prob_churn") >= UMBRAL).cast("double")
)

resultado_lr_ajustado = evaluar_modelo(predictions_lr_ajustado, f"Regresión Logística (umbral {UMBRAL})")


EVALUACIÓN: Regresión Logística (umbral 0.35)
ROC AUC:               0.8486
Accuracy:              0.7853
Precision (Churn=1):   0.5780
Recall (Churn=1):      0.6925   <-- meta del proyecto: > 0.80
F1-Score (Churn=1):    0.6300

Matriz de Confusión:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 1209|
|    0|       1.0|  268|
|    1|       0.0|  163|
|    1|       1.0|  367|
+-----+----------+-----+

¿Cumple Recall > 80%?  ❌ NO


## 7. Tabla comparativa final

In [8]:
# ==============================================================================
# 7. RESUMEN COMPARATIVO DE LOS 3 EXPERIMENTOS
# ==============================================================================
import pandas as pd

resumen = pd.DataFrame([resultado_lr, resultado_rf, resultado_lr_ajustado])
resumen = resumen[["modelo", "roc_auc", "accuracy", "precision", "recall", "f1"]]
resumen.columns = ["Modelo", "ROC AUC", "Accuracy", "Precision (Churn)", "Recall (Churn)", "F1 (Churn)"]
print(resumen.to_string(index=False))

                           Modelo  ROC AUC  Accuracy  Precision (Churn)  Recall (Churn)  F1 (Churn)
 Regresión Logística (umbral 0.5) 0.848620  0.811161           0.683698        0.530189    0.597237
       Random Forest (umbral 0.5) 0.855307  0.808670           0.694149        0.492453    0.576159
Regresión Logística (umbral 0.35) 0.848620  0.785252           0.577953        0.692453    0.630043


## 8. Importancia de variables (Random Forest)
Más allá de "cuántos clientes se van", el equipo de Marketing necesita saber **por qué** se van, para diseñar la campaña de retención correcta. Random Forest calcula internamente qué tan determinante fue cada variable al momento de clasificar, y eso lo podemos extraer directamente.

In [9]:
# ==============================================================================
# 8. RANKING DE VARIABLES MÁS IMPORTANTES PARA PREDECIR CHURN
# ==============================================================================
import pandas as pd

importancias = list(zip(columnas_features, rf_model.featureImportances.toArray()))
importancias_df = pd.DataFrame(importancias, columns=["variable", "importancia"]).sort_values(
    "importancia", ascending=False
).reset_index(drop=True)

print("Top 10 variables que más influyen en el Churn (según Random Forest):")
print(importancias_df.head(10).to_string(index=False))

Top 10 variables que más influyen en el Churn (según Random Forest):
              variable  importancia
        Contract_index     0.209171
                tenure     0.157809
          TotalCharges     0.116275
 InternetService_index     0.085467
        MonthlyCharges     0.076728
  OnlineSecurity_index     0.075585
     TechSupport_index     0.073068
   PaymentMethod_index     0.049772
    OnlineBackup_index     0.022792
PaperlessBilling_index     0.019059


## 9. Exportar resultados para el Dashboard (Fase 4)
Se elige automáticamente, de los tres experimentos evaluados, el que tenga mayor Recall sobre la clase Churn — ya que esa es la métrica objetivo del proyecto. Se exportan dos archivos CSV que la Fase 4 (Dashboard / Power BI) podrá leer directamente:
- **Predicciones por cliente:** `customerID`, si realmente hizo Churn, qué predijo el modelo y con qué probabilidad.
- **Importancia de variables:** el ranking calculado en la sección anterior.

In [10]:
# ==============================================================================
# 9. SELECCIÓN DEL MEJOR MODELO Y EXPORTACIÓN PARA EL DASHBOARD
# ==============================================================================
candidatos = {
    "Regresión Logística (umbral 0.5)": (resultado_lr, predictions_lr),
    "Random Forest (umbral 0.5)": (resultado_rf, predictions_rf),
    f"Regresión Logística (umbral {UMBRAL})": (resultado_lr_ajustado, predictions_lr_ajustado),
}

mejor_nombre = max(candidatos, key=lambda k: candidatos[k][0]["recall"])
mejor_resultado, mejor_predicciones = candidatos[mejor_nombre]

print(f"Modelo seleccionado para el dashboard: {mejor_nombre}")
print(f"Recall: {mejor_resultado['recall']:.4f} | Precision: {mejor_resultado['precision']:.4f}")

# --- Armar la tabla final de predicciones por cliente ---
# Usamos vector_to_array en vez de un UDF de Python por la misma razón de la sección 6
# (evita el PickleException con el tipo Vector de Spark).
predicciones_dashboard = mejor_predicciones.withColumn(
    "probabilidad_churn", vector_to_array(col("probability"))[1]
).select(
    col("customerID"),
    col("label").alias("churn_real"),
    col("prediction").alias("churn_predicho"),
    col("probabilidad_churn")
)

# --- Exportar predicciones a un único archivo CSV con pandas ---
# NOTA: en Windows, predicciones_dashboard.write.csv(...) requiere tener instalado
# "winutils.exe" y la variable de entorno HADOOP_HOME configurada, porque Spark usa la
# API de Hadoop para escribir archivos incluso en disco local. Si no está configurado,
# da el error "Could not locate executable null\bin\winutils.exe in the Hadoop binaries".
# Como esta tabla es pequeña (una fila por cliente del set de prueba), la pasamos a
# pandas y exportamos con to_csv, evitando ese problema por completo.
predicciones_dashboard.toPandas().to_csv(
    "../data/sample/predicciones_churn_dashboard.csv", index=False
)

# --- Exportar importancia de variables ---
importancias_df.to_csv("../data/sample/importancia_variables_churn.csv", index=False)

print("\nArchivos exportados:")
print(" - ../data/sample/predicciones_churn_dashboard.csv (predicciones por cliente)")
print(" - ../data/sample/importancia_variables_churn.csv  (ranking de variables)")

print("\nVista previa de las predicciones a exportar:")
predicciones_dashboard.orderBy(col("probabilidad_churn"), ascending=False).show(10)

Modelo seleccionado para el dashboard: Regresión Logística (umbral 0.35)
Recall: 0.6925 | Precision: 0.5780

Archivos exportados:
 - ../data/sample/predicciones_churn_dashboard.csv (predicciones por cliente)
 - ../data/sample/importancia_variables_churn.csv  (ranking de variables)

Vista previa de las predicciones a exportar:
+----------+----------+--------------+------------------+
|customerID|churn_real|churn_predicho|probabilidad_churn|
+----------+----------+--------------+------------------+
|5178-LMXOP|         1|           1.0|0.8802385866836985|
|6496-SLWHQ|         1|           1.0|0.8783663518337154|
|1400-MMYXY|         1|           1.0|0.8779433243829037|
|4424-TKOPW|         1|           1.0|0.8755797431502518|
|5150-ITWWB|         0|           1.0|0.8708787149021382|
|8098-LLAZX|         1|           1.0|0.8636798817975693|
|3389-YGYAI|         1|           1.0|0.8503721439445638|
|6023-YEBUP|         1|           1.0| 0.849926251591884|
|3988-RQIXO|         1|           

## 🏁 Conclusión
- El **umbral 0.5 por defecto** (en ambos modelos) probablemente se quede corto en Recall, que es justamente la métrica que el proyecto necesita maximizar.
- **Bajar el umbral** es la palanca más rápida para subir el Recall sin reentrenar nada, a costa de algo de Precision (más falsos positivos en la campaña de retención, lo cual es un costo aceptable de negocio).
- Si ninguna de las tres configuraciones llega al 80% de Recall, los siguientes pasos recomendados son: (1) usar `class weight`/`weightCol` en el entrenamiento para que el modelo penalice más los falsos negativos, (2) probar GBTClassifier, o (3) seguir bajando el umbral y validar el impacto en Precision con el equipo de Marketing.
- **Antes de pasar al dashboard:** actualizar la sección 📊 Resultados del README con los números reales de esta tabla, y decidir con el equipo qué modelo/umbral usarán como versión final.
- Los archivos `predicciones_churn_dashboard.csv` y `importancia_variables_churn.csv` ya quedan listos en `data/sample/` para que la Fase 4 (Dashboard) los consuma directamente, sin tener que volver a correr el modelo.